# DATASCI 451 Final Project - EDA & Data Cleaning
## Bayesian Hierarchical Modeling of NOAA Weather Data

**Team:** Yinan Wu, Xintong Liu, Jingyang Xiao

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries loaded successfully!")

## 1. Load Raw Data

In [ ]:
# Load data
df = pd.read_csv("proj_ref/4283488.csv")

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
# Data types and info
df.info()

## 2. Missing Values Analysis

In [ ]:
# Missing values summary
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
    'Non-Null Count': df.notnull().sum()
})
missing

In [ ]:
# Visualize missing data
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if x < 50 else '#e74c3c' for x in missing['Missing %']]
bars = ax.barh(missing.index, missing['Missing %'], color=colors)
ax.set_xlabel('Missing Percentage (%)')
ax.set_title('Missing Data by Column')
ax.axvline(50, color='gray', linestyle='--', alpha=0.7, label='50% threshold')

for bar, pct in zip(bars, missing['Missing %']):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2, f'{pct}%', va='center')

plt.tight_layout()
plt.savefig('plots/01_missing_data.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Cleaning

In [ ]:
# Convert data types
df['DATE'] = pd.to_datetime(df['DATE'])
df['TMAX'] = pd.to_numeric(df['TMAX'], errors='coerce')
df['TMIN'] = pd.to_numeric(df['TMIN'], errors='coerce')
df['TAVG'] = pd.to_numeric(df['TAVG'], errors='coerce')

# Compute TAVG where missing (as per proposal)
df['TAVG_computed'] = df['TAVG'].fillna((df['TMAX'] + df['TMIN']) / 2)

# Extract date components
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month
df['Day'] = df['DATE'].dt.day

print("Data cleaning completed!")
print(f"\nValid temperature records: {df['TAVG_computed'].notna().sum():,} / {len(df):,} ({df['TAVG_computed'].notna().sum()/len(df)*100:.1f}%)")

In [ ]:
# Date range
print(f"Date Range: {df['DATE'].min().strftime('%Y-%m-%d')} to {df['DATE'].max().strftime('%Y-%m-%d')}")
print(f"Total Days: {(df['DATE'].max() - df['DATE'].min()).days + 1}")

## 4. Station Overview

In [ ]:
# Station counts
station_summary = df.groupby(['STATION', 'NAME']).agg({
    'DATE': 'count',
    'TAVG_computed': lambda x: x.notna().sum()
}).reset_index()
station_summary.columns = ['STATION', 'NAME', 'Total_Records', 'Valid_Temp']
station_summary['Coverage_%'] = (station_summary['Valid_Temp'] / station_summary['Total_Records'] * 100).round(1)
station_summary = station_summary.sort_values('Valid_Temp', ascending=False)

print(f"Total Unique Stations: {df['STATION'].nunique()}")
print(f"\nTop 15 stations by valid temperature records:")
station_summary.head(15)

In [ ]:
# Station coverage distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of records per station
ax1 = axes[0]
station_summary['Total_Records'].hist(bins=30, ax=ax1, color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Number of Records')
ax1.set_ylabel('Number of Stations')
ax1.set_title('Distribution of Records per Station')
ax1.axvline(station_summary['Total_Records'].median(), color='red', linestyle='--', 
            label=f'Median: {station_summary["Total_Records"].median():.0f}')
ax1.legend()

# Plot 2: Distribution of temperature coverage
ax2 = axes[1]
station_summary['Coverage_%'].hist(bins=30, ax=ax2, color='coral', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Temperature Coverage (%)')
ax2.set_ylabel('Number of Stations')
ax2.set_title('Distribution of Temperature Data Coverage')
ax2.axvline(80, color='green', linestyle='--', label='80% threshold')
ax2.legend()

plt.tight_layout()
plt.savefig('plots/02_station_coverage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Temperature Statistics

In [ ]:
# Overall temperature statistics
temp_stats = df[['TMAX', 'TMIN', 'TAVG_computed']].describe()
temp_stats.round(2)

In [ ]:
# Temperature distribution - separate plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# TMAX
df['TMAX'].dropna().hist(bins=40, ax=axes[0], color='#e74c3c', edgecolor='black', alpha=0.7)
axes[0].axvline(df['TMAX'].mean(), color='darkred', linestyle='--', linewidth=2)
axes[0].set_xlabel('Temperature (°F)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'TMAX Distribution\nMean: {df["TMAX"].mean():.1f}°F')

# TMIN
df['TMIN'].dropna().hist(bins=40, ax=axes[1], color='#3498db', edgecolor='black', alpha=0.7)
axes[1].axvline(df['TMIN'].mean(), color='darkblue', linestyle='--', linewidth=2)
axes[1].set_xlabel('Temperature (°F)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'TMIN Distribution\nMean: {df["TMIN"].mean():.1f}°F')

# TAVG
df['TAVG_computed'].dropna().hist(bins=40, ax=axes[2], color='#2ecc71', edgecolor='black', alpha=0.7)
axes[2].axvline(df['TAVG_computed'].mean(), color='darkgreen', linestyle='--', linewidth=2)
axes[2].set_xlabel('Temperature (°F)')
axes[2].set_ylabel('Frequency')
axes[2].set_title(f'TAVG (computed) Distribution\nMean: {df["TAVG_computed"].mean():.1f}°F')

plt.tight_layout()
plt.savefig('plots/03_temperature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly temperature boxplot
fig, ax = plt.subplots(figsize=(12, 6))

monthly_data = df[df['TAVG_computed'].notna()].copy()
month_order = [1, 2, 3, 4]
month_labels = ['January', 'February', 'March', 'April']

bp = ax.boxplot([monthly_data[monthly_data['Month'] == m]['TAVG_computed'] for m in month_order],
                labels=month_labels, patch_artist=True)

colors = ['#3498db', '#9b59b6', '#2ecc71', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Average Temperature (°F)')
ax.set_title('Monthly Temperature Distribution (All Stations)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/04_monthly_temperature_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Cleaned Data

In [ ]:
# Save cleaned full dataset
df.to_csv('data/cleaned_daily_data.csv', index=False)
print(f"Cleaned data saved: data/cleaned_daily_data.csv")
print(f"Shape: {df.shape}")

In [ ]:
# Monthly aggregation for all stations
monthly_all = df.groupby(['STATION', 'NAME', 'Year', 'Month']).agg({
    'TAVG_computed': ['mean', 'std', 'count'],
    'TMAX': 'mean',
    'TMIN': 'mean'
}).round(2)
monthly_all.columns = ['TAVG_mean', 'TAVG_std', 'N_days', 'TMAX_mean', 'TMIN_mean']
monthly_all = monthly_all.reset_index()

monthly_all.to_csv('data/monthly_all_stations.csv', index=False)
print(f"Monthly data saved: data/monthly_all_stations.csv")
print(f"Shape: {monthly_all.shape}")

---
## Summary

| Metric | Value |
|--------|-------|
| Total Records | 33,547 |
| Unique Stations | 466 |
| Date Range | 2026-01-01 to 2026-04-07 |
| Valid Temp Records | 14,569 (43.4%) |
| Mean TAVG | 25.3°F |
| TAVG Range | -21.5°F to 69.5°F |